In [2]:
from pyspark.sql import SparkSession, functions as F, types as T

In [3]:
spark = SparkSession.builder.master("local[*]").getOrCreate()

25/06/14 21:51:01 WARN Utils: Your hostname, avillia-AN515-44 resolves to a loopback address: 127.0.1.1; using 192.168.1.102 instead (on interface wlp5s0)
25/06/14 21:51:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/14 21:51:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/14 21:51:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
print(spark.version)

3.5.1


In [4]:
actor_df = spark.read.csv('../data/actor.csv', header=True, inferSchema=True)
address_df = spark.read.csv('../data/address.csv', header=True, inferSchema=True)
category_df = spark.read.csv('../data/category.csv', header=True, inferSchema=True)
city_df = spark.read.csv('../data/city.csv', header=True, inferSchema=True)
country_df = spark.read.csv('../data/country.csv', header=True, inferSchema=True)
customer_df = spark.read.csv('../data/customer.csv', header=True, inferSchema=True)
film_df = spark.read.csv('../data/film.csv', header=True, inferSchema=True)
film_actor_df = spark.read.csv('../data/film_actor.csv', header=True, inferSchema=True)
film_category_df = spark.read.csv('../data/film_category.csv', header=True, inferSchema=True)
inventory_df = spark.read.csv('../data/inventory.csv', header=True, inferSchema=True)
language_df = spark.read.csv('../data/language.csv', header=True, inferSchema=True)
payment_df = spark.read.csv('../data/payment.csv', header=True, inferSchema=True)
rental_df = spark.read.csv('../data/rental.csv', header=True, inferSchema=True)
staff_df = spark.read.csv('../data/staff.csv', header=True, inferSchema=True)
store_df = spark.read.csv('../data/store.csv', header=True, inferSchema=True)

# Домашнє завдання на тему Spark SQL

Задачі з домашнього завдання на SQL потрібно розвʼязати за допомогою Spark SQL DataFrame API.

- Дампи таблиць знаходяться в папці `data`. Датафрейми таблиць вже створені в клітинці вище.
- Можете створювати стільки нових клітинок, скільки вам необхідно.
- Розвʼязок кожної задачі має бути відображений в самому файлі (використати метод `.show()`)
- код має бути оформлений у відповідності із одним із стилем, показаним лектором на занятті 13.

**Увага!**
Використовувати мову запитів SQL безпосередньо забороняється, потрібно використовувати виключно DataFrame API!


1.
Вивести кількість фільмів в кожній категорії.
Результат відсортувати за спаданням.

In [12]:
film_count_by_category_df = film_category_df.groupBy("category_id").count().join(category_df, on="category_id").select("name", "count").show()

+-----------+-----+
|       name|count|
+-----------+-----+
|      Music|   51|
|     Action|   64|
|        New|   63|
|Documentary|   68|
|     Travel|   57|
|   Children|   60|
|     Comedy|   58|
|     Sports|   74|
|    Foreign|   73|
|   Classics|   57|
|     Family|   69|
|      Drama|   62|
|      Games|   61|
|     Horror|   56|
|     Sci-Fi|   61|
|  Animation|   66|
+-----------+-----+



2.
Вивести 10 акторів, чиї фільми брали на прокат найбільше.
Результат відсортувати за спаданням.

In [19]:
top_10_rented_actors_df = rental_df.groupBy("inventory_id").count().join(inventory_df, on="inventory_id").join(film_actor_df, on="film_id").join(actor_df, on="actor_id").orderBy("count", ascending=False).select("first_name", "last_name").show(10, vertical=True)

-RECORD 0--------------
 first_name | LISA     
 last_name  | MONROE   
-RECORD 1--------------
 first_name | OLYMPIA  
 last_name  | PFEIFFER 
-RECORD 2--------------
 first_name | RICHARD  
 last_name  | PENN     
-RECORD 3--------------
 first_name | LUCILLE  
 last_name  | DEE      
-RECORD 4--------------
 first_name | PENELOPE 
 last_name  | CRONYN   
-RECORD 5--------------
 first_name | TOM      
 last_name  | MCKELLEN 
-RECORD 6--------------
 first_name | KEVIN    
 last_name  | BLOOM    
-RECORD 7--------------
 first_name | FRED     
 last_name  | COSTNER  
-RECORD 8--------------
 first_name | CHARLIZE 
 last_name  | DENCH    
-RECORD 9--------------
 first_name | AUDREY   
 last_name  | OLIVIER  
only showing top 10 rows



3.
Вивести категорія фільмів, на яку було витрачено найбільше грошей
в прокаті

In [12]:
most_rented_category_df = rental_df.join(inventory_df, on="inventory_id").join(film_category_df, on="film_id").groupBy("category_id").count().join(category_df, on="category_id").select("name", "count").show(1)

+-----+-----+
| name|count|
+-----+-----+
|Music|  830|
+-----+-----+
only showing top 1 row



4.
Вивести назви фільмів, яких не має в inventory.

In [17]:
films_not_in_inventory_df = film_df.join(inventory_df, on="film_id", how="left_anti").select("title").show()

+--------------------+
|               title|
+--------------------+
|      ALICE FANTASIA|
|         APOLLO TEEN|
|      ARGONAUTS TOWN|
|       ARK RIDGEMONT|
|ARSENIC INDEPENDENCE|
|   BOONDOCK BALLROOM|
|       BUTCH PANTHER|
|       CATCH AMISTAD|
| CHINATOWN GLADIATOR|
|      CHOCOLATE DUCK|
|COMMANDMENTS EXPRESS|
|    CROSSING DIVORCE|
|     CROWDS TELEMARK|
|    CRYSTAL BREAKING|
|          DAZED PUNK|
|DELIVERANCE MULHO...|
|   FIREHOUSE VIETNAM|
|       FLOATS GARDEN|
|FRANKENSTEIN STRA...|
|  GLADIATOR WESTWARD|
+--------------------+
only showing top 20 rows



5.
Вивести топ 3 актори, які найбільше зʼявлялись в категорії фільмів “Children”

In [23]:
top_3_children_actors_df = (
    actor_df
    .join(film_actor_df, on="actor_id")
    .join(film_category_df, on="film_id")
    .join(category_df, on="category_id")
    .where("name = 'Children'")
    .groupBy("actor_id", "first_name", "last_name")
    .count()
    .orderBy("count", ascending=False)
    .show(3)
)


+--------+----------+---------+-----+
|actor_id|first_name|last_name|count|
+--------+----------+---------+-----+
|      17|     HELEN|   VOIGHT|    7|
|     127|     KEVIN|  GARLAND|    5|
|      80|     RALPH|     CRUZ|    5|
+--------+----------+---------+-----+
only showing top 3 rows



Stop Spark session:

In [24]:
spark.stop()